In [16]:
# Loading libraries
suppressPackageStartupMessages({
  library(ENmix)
  library(IlluminaHumanMethylationEPICmanifest)
  library(IlluminaHumanMethylationEPICanno.ilm10b2.hg19)
  library(parallel)
  library(methylCIPHER)
    library(glmnet);
    library(EpiDISH);
    library(dnaMethyAge);
    library(prcPhenoAge);
    library(DunedinPoAm38)
})

In [2]:
# Reading phenotypic data of DNAm in UTHealth cohort
mf = "/gpfs/gibbs/pi/montalvo-ortiz/epigenomics/braindata/collaborations/consuelobassdata/brain_dataset/meta_BA9_brainbank_methylation_sub_3-29-22.csv"
mf = read.csv(mf)
mf$name <- paste0("BRA_", mf$Sentrix_ID)
head(mf$name)

[1] "BRA_201465920012_R01C01" "BRA_201465920012_R02C01"
[3] "BRA_201465920012_R03C01" "BRA_201465920012_R04C01"
[5] "BRA_201465920012_R05C01" "BRA_201465920012_R06C01"

In [3]:
load("/vast/palmer/scratch/montalvo-ortiz/jjm262/00tclocks/00uthealth/00databases/05epigenome/00uthealth_brain-07312025.RData")
load("/vast/palmer/scratch/montalvo-ortiz/jjm262/00tclocks/00uthealth/00databases/05epigenome/00uthealth_brain-07302025.RData")

In [4]:
# setting cores
num_threads <- detectCores() - 1
print(num_threads)
# set working directory
wkdir="/vast/palmer/scratch/montalvo-ortiz/jjm262/00tclocks/00uthealth/00databases/05epigenome"
setwd(wkdir)
ls()

[1] 47


[1] "beta"        "mdat"        "mf"          "num_threads" "pheno"      
[6] "qc"          "wkdir"

In [5]:
# Perform SVA analysis
path="/gpfs/gibbs/pi/montalvo-ortiz/epigenomics/braindata/collaborations/consuelobassdata/brain_dataset/idat_brain"
suppressMessages({
  rgSet <- readidat(path = path, recursive = TRUE, force = TRUE)
})
sva <- ctrlsva(rgSet)
sva = as.data.frame(sva)
sva$name = rownames(sva)

BRA_201465920012_R01C01 BRA_201465920012_R02C01 BRA_201465920012_R03C01 
                1051943                 1051943                 1051943 
BRA_201465920012_R04C01 BRA_201465920012_R05C01 BRA_201465920012_R06C01 
                1051943                 1051943                 1051943 
BRA_201465920012_R07C01 BRA_201465920012_R08C01 BRA_201465920048_R01C01 
                1051943                 1051943                 1051943 
BRA_201465920048_R02C01 BRA_201465920048_R03C01 BRA_201465920048_R04C01 
                1051943                 1051943                 1051943 
BRA_201465920048_R05C01 BRA_201465920048_R06C01 BRA_201465920048_R07C01 
                1051943                 1051943                 1051943 
BRA_201465920048_R08C01 BRA_201532190145_R01C01 BRA_201532190145_R02C01 
                1051943                 1051815                 1051815 
BRA_201532190145_R03C01 BRA_201532190145_R04C01 BRA_201532190145_R05C01 
                1051815                 1051815    

In [7]:
celltype = estimateCellProp(userdata=rgSet,
         refdata="FlowSorted.DLPFC.450k",
         nonnegative = TRUE, normalize=TRUE)

In [8]:
# Merge celltypes
celltype$name = celltype$Sample_Name
pheno = merge(mf, celltype, by = c('name'))
pheno = merge(pheno, sva, by = c('name'))

In [9]:
# First estimate Betas
beta = getB(mdat)

In [10]:
# Subsetting samples to those that pass QC
pheno_sub = pheno[pheno$Sample_Name %in% colnames(beta),]

In [11]:
# Estimate clocks using methylCIPHER
# Need to transpose the matrix, samples in columns and cpg in rows
beta = as.data.frame(t(beta))
pheno_sub = calcCoreClocks(beta, pheno_sub)

Please remember to cite the core Clocks you have used! Please refer to the README.md file for assistance.



In [19]:
prcPhenoAge::
clockOptions()

[1] "calcAlcoholMcCartney"            "calcBMIMcCartney"               
 [3] "calcBocklandt"                   "calcBohlin"                     
 [5] "calcClockCategory"               "calcDNAmClockCortical"          
 [7] "calcDNAmTL"                      "calcDunedinPoAm38"              
 [9] "calcEpiTOC"                      "calcEpiTOC2"                    
[11] "calcGaragnani"                   "calcHannum"                     
[13] "calcHorvath1"                    "calcHorvath2"                   
[15] "calcHRSInChPhenoAge"             "calcHypoClock"                  
[17] "calcKnight"                      "calcLeeControl"                 
[19] "calcLeeRefinedRobust"            "calcLeeRobust"                  
[21] "calcLin"                         "calcMayne"                      
[23] "calcMiAge"                       "calcPEDBE"                      
[25] "calcPhenoAge"                    "calcSmokingMcCartney"           
[27] "calcVidalBralo"                  "calcWeidner"                    
[29] "calcZhang"                       "calcZhang2019"                  
[31] "prcPhenoAge::calcPRCPhenoAge"    "prcPhenoAge::calcnonPRCPhenoAge"
[33] "DunedinPoAm38::PoAmProjector"

In [25]:
# Estimate Zhang
userClocks = c("calcZhang2019")
pheno_sub = calcUserClocks(userClocks, beta, pheno_sub, imputation = FALSE)

In [26]:
# Preparing data for stochastic clock
beta_m = as.matrix(t(beta))
age = as.vector(pheno_sub$Age)
dim(beta_m)
dim(age)
head(colnames(beta_m))  # Should be sample IDs

[1] 821117    114

NULL

[1] "BRA_201465920012_R01C01" "BRA_201465920012_R02C01"
[3] "BRA_201465920012_R03C01" "BRA_201465920012_R04C01"
[5] "BRA_201465920012_R05C01" "BRA_201465920012_R06C01"

In [27]:
# Running stocastic clocks
clockpath = "/vast/palmer/scratch/montalvo-ortiz/jjm262/00tclocks/00uthealth/03scripts/01epigenome/glmStocALL.Rd"

# This is the function for stochastic clocks
RunStochClocks <- function(data.m,ages.v=NULL,refM.m=NULL){
 load(clockpath); ## load in stochastic clock information
 estF.m <- NULL;
 if(!is.null(refM.m)){
   estF.m <- epidish(data.m,ref=refM.m,method="RPC",maxit=500)$est;
 }
 predMage.lv <- list();
 eaa.lv <- list();
 iaa.lv <- list();
 for(c in 1:length(glmStocALL.lo)){
  glm.o <- glmStocALL.lo[[c]];
  coef.m <- coef(glm.o);
  intC <- coef.m[1,ncol(coef.m)]; ### intercept term
  coef.v <- coef.m[-1,ncol(coef.m)]; ### estimated regression coefficients
  commonCpGs.v <- intersect(names(coef.v),rownames(data.m));
  rep.idx <- match(commonCpGs.v,names(coef.v));
  map.idx <- match(commonCpGs.v,rownames(data.m));
  predMage.lv[[c]] <- as.vector(intC + matrix(coef.v[rep.idx],nrow=1) %*% data.m[map.idx,]);
  if(!is.null(ages.v)){
     eaa.lv[[c]] <- lm(predMage.lv[[c]] ~ ages.v)$res;     
     if(!is.null(refM.m)){
      iaa.lv[[c]] <- lm(predMage.lv[[c]] ~ ages.v + estF.m)$res;
     }
  }
 }
 return(list(mage=predMage.lv,eaa=eaa.lv,iaa=iaa.lv,estF=estF.m));
}

# Running function
sclocks = RunStochClocks(beta_m, age)

In [28]:
# Add sample IDs first (these are the column names of beta)
sample_ids <- colnames(beta_m)

mage_df <- as.data.frame(do.call(cbind, sclocks$mage))
colnames(mage_df) <- paste0("StochClock", seq_along(sclocks$mage))
mage_df$name <- sample_ids

eaa_df <- as.data.frame(do.call(cbind, sclocks$eaa))
colnames(eaa_df) <- paste0("EAA_StochClock", seq_along(sclocks$eaa))
eaa_df$name <- sample_ids

# Merge side by side by 'name'
combined_df <- merge(mage_df, eaa_df, by = "name")
combined_df
pheno_sub = merge(pheno_sub, combined_df, by = c('name'))
pheno_sub

name,StochClock1,StochClock2,StochClock3,EAA_StochClock1,EAA_StochClock2,EAA_StochClock3
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
BRA_201465920012_R01C01,21.84555,4.513590,38.72373,-3.1664033,-0.9963250,-5.1253923
BRA_201465920012_R02C01,17.80645,7.576214,45.29077,-2.7238284,9.4132725,2.2618458
BRA_201465920012_R03C01,29.46791,1.795859,39.73101,1.0946989,-9.2242857,-4.7332627
BRA_201465920012_R04C01,16.31163,2.873425,45.25659,-5.7125354,2.2614928,1.9542652
BRA_201465920012_R05C01,26.86126,3.694324,42.52529,0.3554117,-4.2645822,-1.5972339
BRA_201465920012_R06C01,17.75092,9.953468,46.16633,-10.2488188,-0.4544291,1.7704115
BRA_201465920012_R08C01,17.14162,8.469922,40.10796,-3.7621331,9.6947330,-2.9893153
BRA_201465920048_R01C01,28.24137,8.600238,36.47719,0.9885796,-0.5831638,-7.7820366
BRA_201465920048_R02C01,36.17728,26.616832,43.90820,4.8162826,10.6987049,-1.1028696


name,SAB,Sentrix_ID,Year,Ethnicity,Gender,Age,PMIhrs,pH,Cause_of_Death,⋯,Horvath1,PhenoAge,epiTOC2,Zhang2019,StochClock1,StochClock2,StochClock3,EAA_StochClock1,EAA_StochClock2,EAA_StochClock3
<chr>,<int>,<chr>,<int>,<chr>,<chr>,<int>,<dbl>,<dbl>,<chr>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
BRA_201465920012_R01C01,67917,201465920012_R01C01,2018,Hispanic,Male,44,32.67,5.91,acute carisoprodol toxicity,⋯,39.30907,-22.791375,181.13486,30.09624,21.84555,4.513590,38.72373,-3.1664033,-0.9963250,-5.1253923
BRA_201465920012_R02C01,67927,201465920012_R02C01,2018,Black,Female,32,31.57,6.56,undetermined,⋯,39.53668,-17.402457,62.77680,25.51929,17.80645,7.576214,45.29077,-2.7238284,9.4132725,2.2618458
BRA_201465920012_R03C01,67928,201465920012_R03C01,2018,Black,Male,53,28.65,6.79,acute toxicity of cocaine and ethanol,⋯,46.63943,-20.579133,369.31879,33.18496,29.46791,1.795859,39.73101,1.0946989,-9.2242857,-4.7332627
BRA_201465920012_R04C01,67929,201465920012_R04C01,2018,White,Female,36,28.20,6.80,toxic effects of methadone,⋯,34.85467,-20.439521,901.55171,30.34616,16.31163,2.873425,45.25659,-5.7125354,2.2614928,1.9542652
BRA_201465920012_R05C01,67930,201465920012_R05C01,2018,White,Male,48,31.30,6.44,GSW,⋯,43.41878,-30.526159,946.82365,40.70224,26.86126,3.694324,42.52529,0.3554117,-4.2645822,-1.5972339
BRA_201465920012_R06C01,67931,201465920012_R06C01,2018,White,Male,52,31.92,6.38,"cardiovascular disease, not opioid",⋯,51.33423,-21.974027,917.82742,41.69750,17.75092,9.953468,46.16633,-10.2488188,-0.4544291,1.7704115
BRA_201465920012_R08C01,67943,201465920012_R08C01,2018,White,Male,33,10.57,6.72,"multi overdose: benzodiazepine, amphetamine, cocaine, EtOH, and opioid",⋯,36.35575,-20.421391,456.39942,24.81313,17.14162,8.469922,40.10796,-3.7621331,9.6947330,-2.9893153
BRA_201465920048_R01C01,67944,201465920048_R01C01,2018,White,Female,50,9.55,6.63,sepsis,⋯,50.16851,-22.062699,838.08977,37.18442,28.24137,8.600238,36.47719,0.9885796,-0.5831638,-7.7820366
BRA_201465920048_R02C01,67945,201465920048_R02C01,2018,Black,Male,61,37.40,6.64,Cardiovascular disease,⋯,63.10395,-9.884480,1429.80421,43.36969,36.17728,26.616832,43.90820,4.8162826,10.6987049,-1.1028696


In [29]:
# Save specific objects to an RData file
save(beta, qc, pheno, pheno_sub, rgSet, file = "/vast/palmer/scratch/montalvo-ortiz/jjm262/00tclocks/00uthealth/00databases/05epigenome/00uthealth_brain-08022025.RData")

In [30]:
# Save specific objects to an RData file
saveRDS(pheno_sub, file = "/vast/palmer/scratch/montalvo-ortiz/jjm262/00tclocks/00uthealth/00databases/05epigenome/00uthealth_brain-08022025.rds")

In [31]:
# Modification made on 08032025
load("/vast/palmer/scratch/montalvo-ortiz/jjm262/00tclocks/00uthealth/00databases/05epigenome/00uthealth_brain-08022025.RData")

In [32]:
# Estimate cortical clock
cortical_age <- methyAge(t(beta), clock='ShirebyG2020')

Warning message in methyAge(t(beta), clock = "ShirebyG2020"):
“Found  6 out of 348 probes missing! They will be assigned with mean values from reference dataset, missing probes are:
  cg00771642 cg09372546 cg10904972 cg22285878 cg24346776 cg24420164”


In [33]:
# Change colnames
colnames(cortical_age) = c("name", "corticalClock")
# Merge
pheno_sub = merge(pheno_sub, cortical_age, by = c('name'))
pheno_sub

name,SAB,Sentrix_ID,Year,Ethnicity,Gender,Age,PMIhrs,pH,Cause_of_Death,⋯,PhenoAge,epiTOC2,Zhang2019,StochClock1,StochClock2,StochClock3,EAA_StochClock1,EAA_StochClock2,EAA_StochClock3,corticalClock
<chr>,<int>,<chr>,<int>,<chr>,<chr>,<int>,<dbl>,<dbl>,<chr>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
BRA_201465920012_R01C01,67917,201465920012_R01C01,2018,Hispanic,Male,44,32.67,5.91,acute carisoprodol toxicity,⋯,-22.791375,181.13486,30.09624,21.84555,4.513590,38.72373,-3.1664033,-0.9963250,-5.1253923,40.40864
BRA_201465920012_R02C01,67927,201465920012_R02C01,2018,Black,Female,32,31.57,6.56,undetermined,⋯,-17.402457,62.77680,25.51929,17.80645,7.576214,45.29077,-2.7238284,9.4132725,2.2618458,38.11405
BRA_201465920012_R03C01,67928,201465920012_R03C01,2018,Black,Male,53,28.65,6.79,acute toxicity of cocaine and ethanol,⋯,-20.579133,369.31879,33.18496,29.46791,1.795859,39.73101,1.0946989,-9.2242857,-4.7332627,44.07033
BRA_201465920012_R04C01,67929,201465920012_R04C01,2018,White,Female,36,28.20,6.80,toxic effects of methadone,⋯,-20.439521,901.55171,30.34616,16.31163,2.873425,45.25659,-5.7125354,2.2614928,1.9542652,34.78782
BRA_201465920012_R05C01,67930,201465920012_R05C01,2018,White,Male,48,31.30,6.44,GSW,⋯,-30.526159,946.82365,40.70224,26.86126,3.694324,42.52529,0.3554117,-4.2645822,-1.5972339,46.06222
BRA_201465920012_R06C01,67931,201465920012_R06C01,2018,White,Male,52,31.92,6.38,"cardiovascular disease, not opioid",⋯,-21.974027,917.82742,41.69750,17.75092,9.953468,46.16633,-10.2488188,-0.4544291,1.7704115,47.65431
BRA_201465920012_R08C01,67943,201465920012_R08C01,2018,White,Male,33,10.57,6.72,"multi overdose: benzodiazepine, amphetamine, cocaine, EtOH, and opioid",⋯,-20.421391,456.39942,24.81313,17.14162,8.469922,40.10796,-3.7621331,9.6947330,-2.9893153,36.07565
BRA_201465920048_R01C01,67944,201465920048_R01C01,2018,White,Female,50,9.55,6.63,sepsis,⋯,-22.062699,838.08977,37.18442,28.24137,8.600238,36.47719,0.9885796,-0.5831638,-7.7820366,42.25234
BRA_201465920048_R02C01,67945,201465920048_R02C01,2018,Black,Male,61,37.40,6.64,Cardiovascular disease,⋯,-9.884480,1429.80421,43.36969,36.17728,26.616832,43.90820,4.8162826,10.6987049,-1.1028696,61.40346


In [34]:
# Save specific objects to an RData file
saveRDS(pheno_sub, file = "/vast/palmer/scratch/montalvo-ortiz/jjm262/00tclocks/00uthealth/00databases/05epigenome/00uthealth_brain-08032025.rds")

In [35]:
# Save Pheno and Beta in .CSV files
write.table(pheno_sub, file = "/vast/palmer/scratch/montalvo-ortiz/jjm262/00tclocks/00uthealth/00databases/05epigenome/00pheno_sub-08042025.txt",
          sep = '\t',
          quote = FALSE,
          row.names = FALSE)

In [7]:
# Round to three decimals
beta_rounded <- round(beta, 3)
# Save to CSV
write.csv(beta_rounded, file = "/vast/palmer/scratch/montalvo-ortiz/jjm262/00tclocks/00uthealth/00databases/05epigenome/01beta_rounded-08042025.csv", quote = FALSE, row.names = TRUE)